In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from tqdm import tqdm 
sys.path.append('../../')
from utils.memory_utils import reduce_mem_usage, process_in_chunks, save_in_chunks

user_data = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_CDs_and_Vinyl", split="full", trust_remote_code=True)
meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_CDs_and_Vinyl", split="full", trust_remote_code=True)

def flatten_lst(col):
    # flatten lists by joining their elements with a separator (e.g., ', ')
    col['features'] = ', '.join(col['features']) if isinstance(col['features'], list) else col['features']
    col['description'] = ', '.join(col['description']) if isinstance(col['description'], list) else col['description']
    col['categories'] = ', '.join(col['categories']) if isinstance(col['categories'], list) else col['categories']

    return col

flattened_meta = meta.map(flatten_lst)

user_df = process_in_chunks(user_data.to_pandas(), chunk_size=10000)
meta_df = process_in_chunks(flattened_meta.to_pandas(), chunk_size=10000)

# user_df = user_data.to_pandas().fillna(0)
# meta_df = flattened_meta.to_pandas().fillna(0)

Processed 0 rows. Current memory usage: 4258.69 MB
Processed 10000 rows. Current memory usage: 4262.08 MB
Processed 20000 rows. Current memory usage: 4265.14 MB
Processed 30000 rows. Current memory usage: 4268.83 MB
Processed 40000 rows. Current memory usage: 4272.70 MB
Processed 50000 rows. Current memory usage: 4276.45 MB
Processed 60000 rows. Current memory usage: 4279.92 MB
Processed 70000 rows. Current memory usage: 4283.27 MB
Processed 80000 rows. Current memory usage: 4289.81 MB
Processed 90000 rows. Current memory usage: 4293.22 MB
Processed 100000 rows. Current memory usage: 4296.72 MB
Processed 110000 rows. Current memory usage: 4300.36 MB
Processed 120000 rows. Current memory usage: 4303.41 MB
Processed 130000 rows. Current memory usage: 4306.48 MB
Processed 140000 rows. Current memory usage: 4309.56 MB
Processed 150000 rows. Current memory usage: 4311.83 MB
Processed 160000 rows. Current memory usage: 4315.06 MB
Processed 170000 rows. Current memory usage: 4317.55 MB
Proces

In [2]:
user_df['verified_purchase'] = user_df['verified_purchase'].astype(int)
user_df.head(5)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,LOVE IT!,[],B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,0,1
1,5.0,Five Stars,LOVE!!,[],B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,0,1
2,3.0,Three Stars,Sad there is not the versions with the real/or...,[],B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,0,1
3,3.0,Disappointed,I have listen to The Broadway 1958 Flower Drum...,[],B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,3,1
4,5.0,Wonderful melding,Simply great album. One of the best. Marvelous...,[],B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,0,0


In [3]:
user_df['rating'] = user_df['rating'].astype('float32')
user_df.groupby(['rating','verified_purchase']).size().reset_index().pivot(columns='rating', index='verified_purchase', values=0)

rating,1.0,2.0,3.0,4.0,5.0
verified_purchase,,,,,
0,81604,63704,117798,288213,996369
1,106123,76853,165893,376301,2554415


In [4]:
user_df2 = user_df[['rating', 'asin', 'parent_asin', 'user_id', 'timestamp', 'text']]
user_df2

,rating,asin,parent_asin,user_id,timestamp,text
0,5.0,B002MW50JA,B002MW50JA,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650777000,LOVE IT!
1,5.0,B008XNPN0S,B008XNPN0S,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650764000,LOVE!!
2,3.0,B00IKM5N02,B00IKM5N02,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452649885000,Sad there is not the versions with the real/or...
3,3.0,B00006JKCM,B00006JKCM,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1164036864000,I have listen to The Broadway 1958 Flower Drum...
4,5.0,B00013YRQY,B00013YRQY,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1582090199946,Simply great album. One of the best. Marvelous...
...,...,...,...,...,...,...
4827268,5.0,B000002VPH,B000002VPH,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308413175000,I love this cd and love the movie thank u I ha...
4827269,5.0,B000084T18,B000084T18,AHM36UEBOF2I6VH7CGAGHCDDUITQ,1308412875000,I love the cd it play real well and was delive...
4827270,5.0,B004OFWLO0,B004OFWLO0,AHRJPHOI5JHYEQVSDMNX6736QH3Q,1505425559120,Such a great remaster you can fully appreciate...
4827271,1.0,B000GIXIAK,B000GIXIAK,AH4PJ73QN75AJM5VSCT53AOADCGA,1157470110000,What this CD is lacking is a heart. The music...


In [5]:
meta_df.head(3)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Digital Music,Release Some Tension,4.601562,112,,Swv ~ Release Some Tension,12.05,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SWV Format: Audio CD,"CDs & Vinyl, Dance & Electronic, House","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B000002X4C,None,None,None
1,Digital Music,Rio Angie,5.000000,1,,"Shrimp City Slim (aka Gary Erwin, b. 1953, Chi...",14.98,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Shrimp City Slim (Artist) Format: Audio CD,"CDs & Vinyl, Jazz, Avant Garde & Free Jazz","{""Product Dimensions"": ""5.6 x 0.4 x 4.9 inches...",B00902T10Y,None,None,None
2,Digital Music,Lost in Love,5.000000,9,,,24.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Nastyboy Klick Format: Audio CD,"CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore","{""Package Dimensions"": ""4.7 x 4.6 x 0.1 inches...",B00000DALY,None,None,None


In [6]:
meta_df['main_category'].unique()
meta_df.shape

(701959, 16)

In [7]:
meta_df = meta_df.drop_duplicates(subset=['parent_asin'],keep='last')
meta_df.shape

(701959, 16)

In [8]:
# drop cols dont provide contextual meaning
meta = meta_df.drop(['images', 'features', 'videos', 'store', 'price', 'details', 'bought_together', 'subtitle', 'author'], axis='columns')
meta

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,main_category,title,average_rating,rating_number,description,categories,parent_asin
0,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",B000002X4C
1,Digital Music,Rio Angie,5.000000,1,"Shrimp City Slim (aka Gary Erwin, b. 1953, Chi...","CDs & Vinyl, Jazz, Avant Garde & Free Jazz",B00902T10Y
2,Digital Music,Lost in Love,5.000000,9,,"CDs & Vinyl, Rap & Hip-Hop, Gangsta & Hardcore",B00000DALY
3,Digital Music,Somewhere in Time,4.800781,1186,The 1980 soundtrack to the now classic motion ...,"CDs & Vinyl, Soundtracks, Movie Scores",B0000086D1
4,Digital Music,Kimmon Waldruff,5.000000,1,Solo acoustic fingerstyle guitar.,"CDs & Vinyl, Folk",B000S6W7KC
...,...,...,...,...,...,...,...
701954,Digital Music,Forever Gold: British Invasion,3.000000,1,British Invasion ~ Forever Gold: British Invasion,"CDs & Vinyl, International Music, Europe, Brit...",B00005AVHN
701955,Digital Music,"Joan Hammond, Historical Recordings from 1941-49",5.000000,1,,"CDs & Vinyl, Classical",B000T001IM
701956,Digital Music,Come Alive,4.500000,4,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout...",B00069I6RO
701957,Digital Music,Long Day in the Milky Way,4.398438,16,2020 release from the folk/Americana singer/so...,"CDs & Vinyl, Folk",B08BF2PH1X


In [9]:
merge = user_df2.merge(meta, on='parent_asin', how='right')
merge = merge.dropna()
merge

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,asin,parent_asin,user_id,timestamp,text,main_category,title,average_rating,rating_number,description,categories
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Definitely the weakest of the three albums SWV...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Unlike It's About Time and New Beginning this ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,"Their third and final album as a group, &quot;...",Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
3,5.0,B000002X4C,B000002X4C,AE73R3KLFKXFNWUBOA4JCLT5UWKA,1.427480e+12,thank you,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
4,2.0,B000002X4C,B000002X4C,AFICHVZ62IFAWIBZK3U6B7NZWAVQ,1.289638e+12,When I was trying to decide whether or not to ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House"
...,...,...,...,...,...,...,...,...,...,...,...,...
4827554,5.0,B000T001IM,B000T001IM,AFICGLJ62Y3YRTYN7PNYM6UUOA6Q,1.387540e+12,"Dame Joan Hilda Hood Hammond, DBE, CMG (24 May...",Digital Music,"Joan Hammond, Historical Recordings from 1941-49",5.000000,1,,"CDs & Vinyl, Classical"
4827555,4.0,B00069I6RO,B00069I6RO,AHLRFMDIVAQF7ARTPM6PVJO6DTYQ,1.199348e+12,I first saw Heinz Winckler in a Broadway music...,Digital Music,Come Alive,4.500000,4,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout..."
4827556,5.0,B00069I6RO,B00069I6RO,AGZVQUGBE64F3M2A5EBU2CNDTKIA,1.103150e+12,"OK, loves, it's cheaper to go to amazon.co.uk ...",Digital Music,Come Alive,4.500000,4,The Second Full Length Album from the Winner o...,"CDs & Vinyl, International Music, Africa, Sout..."
4827557,5.0,B08BF2PH1X,B08BF2PH1X,AEZET6CFIEJWTU7UTEK57IDARX2A,1.612912e+12,Wonderful collection of songs with haunting tu...,Digital Music,Long Day in the Milky Way,4.398438,16,2020 release from the folk/Americana singer/so...,"CDs & Vinyl, Folk"


In [10]:
merge.columns = [
    'rating', 'iid', 'parent_iid', 'uid', 'timestamp', 'reviews',
    'main_category', 'title', 'average_rating', 'rating_number',
    'description', 'categories'
]

In [11]:
merge.iid.nunique(), merge.parent_iid.unique(), merge.uid.nunique()

(700877,
 array(['B000002X4C', 'B00902T10Y', 'B00000DALY', ..., 'B00069I6RO',
        'B08BF2PH1X', 'B000001LW6'], shape=(700844,), dtype=object),
 1752912)

In [12]:
merge_df = merge.copy()

In [13]:
item_info = merge_df.groupby('iid').agg({"rating_number": 'first'})
user_info = merge_df.groupby('uid').agg({"rating":'count'})

In [14]:
# filters the user_info, item_info DF to select items that have received more than 5 ratings
active_item = item_info[item_info['rating_number']>5].index
active_user = user_info[user_info['rating']>5].index
active_item.shape, active_user.shape

((452351,), (142289,))

In [15]:
merge_df = merge[merge['uid'].isin(active_user) & merge['iid'].isin(active_item)]

In [16]:
# assign binary label (implicit feedback style)
merge_df['label'] = 0
merge_df.loc[merge_df['rating'] >= 4, 'label'] = 1

/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_42143/3053501079.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df['label'] = 0


In [31]:
# sample interactions for scalability (10k or 100k rows)
sample_df = merge_df.sample(n=100000, random_state=42)

In [33]:
# def leave_one_out_with_sampling(df, test_sample_ratio=0.7, cold_start_ratio=None, min_user_interactions=2):
#     """
#     Simple approach:
#     1. Do leave-one-out split for users with >=2 interactions
#     2. Keep only 70% of the LOO test data
#     3. Add 10% of cold-start users (1 interaction) to test set
#     """
    
#     # Sort by timestamp
#     df = df.sort_values(['uid', 'timestamp'])
    
#     # Step 1: Separate users by interaction count
#     user_counts = df['uid'].value_counts()
    
#     # Qualified users (for leave-one-out)
#     qualified_users = user_counts[user_counts >= min_user_interactions].index
#     qualified_df = df[df['uid'].isin(qualified_users)].copy()
    
#     # Cold-start users (only 1 interaction)
#     cold_start_users = user_counts[user_counts == 1].index
#     cold_start_df = df[df['uid'].isin(cold_start_users)].copy()
    
#     print(f"Total users: {len(user_counts)}")
#     print(f"Qualified users (≥{min_user_interactions} interactions): {len(qualified_users)}")
#     print(f"Cold-start users (1 interaction): {len(cold_start_users)}")
    
#     # Step 2: Leave-one-out split for qualified users
#     test_ = qualified_df.groupby('uid').tail(1).copy()
#     train_ = qualified_df[~qualified_df.index.isin(test_.index)].copy()
    
#     print(f"\nLeave-one-out results:")
#     print(f"LOO train: {len(train_)} interactions")
#     print(f"LOO test: {len(test_)} interactions")
    
#     # Step 3: Sample 70% of test_ data
#     test_sampled = test_.sample(frac=test_sample_ratio, random_state=42)
#     test_remaining = test_[~test_.index.isin(test_sampled.index)]
    
#     print(f"Sampled test 70%: {len(test_sampled)} interactions")
#     print(f"Returned to train: {len(test_remaining)} interactions")
    
#     # Step 4: Sample 5% of cold-start users for test
#     n_cold_start_test = int(len(cold_start_users) * cold_start_ratio)
#     cold_start_test_users = cold_start_users[:n_cold_start_test]  
#     cold_start_test = cold_start_df[cold_start_df['uid'].isin(cold_start_test_users)].copy()
#     cold_start_train = cold_start_df[~cold_start_df['uid'].isin(cold_start_test_users)].copy()
    
#     print(f"\nCold-start sampling:")
#     print(f"Cold-start test users: {len(cold_start_test_users)}")
#     print(f"Cold-start test interactions: {len(cold_start_test)}")
#     print(f"Cold-start train interactions: {len(cold_start_train)}")
    
#     # Step 5: Combine everything
#     train_data = pd.concat([train_, test_remaining, cold_start_train], ignore_index=True)
#     test_data = pd.concat([test_sampled, cold_start_test], ignore_index=True)
    
#     # Add cold-start indicator
#     test_data['cold_start'] = test_data['uid'].isin(cold_start_test_users).astype(int)
    
#     # Final statistics
#     print(f"\nFINAL SPLIT RESULTS:")
#     print(f"Train: {train_data.shape[0]} interactions, {train_data['uid'].nunique()} users")
#     print(f"Test: {test_data.shape[0]} interactions, {test_data['uid'].nunique()} users")
    
#     cold_start_test = test_data[test_data['cold_start'] == 1]
#     warm_start_test = test_data[test_data['cold_start'] == 0]
    
#     print(f"\nTest set composition:")
#     print(f"Warm-start test (from LOO): {len(warm_start_test)} interactions")
#     print(f"Cold-start test: {len(cold_start_test)} interactions")
#     print(f"Cold-start ratio in test: {len(cold_start_test)/len(test_data):.1%}")
    
#     return train_data, test_data

In [34]:
 # train_data, test_data = leave_one_out_with_sampling(
 #        sample_df,
 #        test_sample_ratio=0.7,    
 #        cold_start_ratio=0.02,     
 #        min_user_interactions=2
 #    )

Total users: 8869
Qualified users (≥2 interactions): 820
Cold-start users (1 interaction): 8049

Leave-one-out results:
LOO train: 1131 interactions
LOO test: 820 interactions
Sampled test 70%: 574 interactions
Returned to train: 246 interactions

Cold-start sampling:
Cold-start test users (10%): 160
Cold-start test interactions: 160
Cold-start train interactions: 7889

FINAL SPLIT RESULTS:
Train: 9266 interactions, 8709 users
Test: 734 interactions, 734 users

Test set composition:
Warm-start test (from LOO): 574 interactions
Cold-start test: 160 interactions
Cold-start ratio in test: 21.8%


In [32]:
user_counts = sample_df['uid'].value_counts()
min_user_interactions = 2
qualified_users = user_counts[user_counts >= min_user_interactions].index
filtered_df = sample_df[sample_df['uid'].isin(qualified_users)].copy()
print(f"After filtering users with >= {min_user_interactions} interactions: {len(filtered_df)} interactions")
print(f"Qualified users: {len(qualified_users)}")

After filtering users with >= 2 interactions: 61081 interactions
Qualified users: 19080


In [33]:
# Sort by timestamp to ensure we take the most recent interaction
filtered_df = filtered_df.sort_values(['uid', 'timestamp'])

# Leave-One-Out split: last interaction per user for test
test_data = filtered_df.groupby('uid').tail(1).copy()
train_data = filtered_df[~filtered_df.index.isin(test_data.index)].copy()
print(f"\nLeave-One-Out Split Results:")
print(f"Train: {train_data.shape[0]} interactions, {train_data['uid'].nunique()} users")
print(f"Test: {test_data.shape[0]} interactions, {test_data['uid'].nunique()} users")


Leave-One-Out Split Results:
Train: 42001 interactions, 19080 users
Test: 19080 interactions, 19080 users


In [34]:
# Check item overlap (this should be much better now)
train_items = set(train_data['iid'].unique())
test_items = set(test_data['iid'].unique())
overlap = train_items.intersection(test_items)
print(f"Item overlap: {len(overlap)}/{len(test_items)} ({len(overlap)/len(test_items):.1%})")
    
# Additional diagnostics
print(f"\nAdditional Diagnostics:")
print(f"Average interactions per user (train): {train_data.shape[0] / train_data['uid'].nunique():.1f}")
print(f"Average interactions per user (test): {test_data.shape[0] / test_data['uid'].nunique():.1f}")

Item overlap: 4780/16475 (29.0%)

Additional Diagnostics:
Average interactions per user (train): 2.2
Average interactions per user (test): 1.0


In [35]:
# Check if any users lost all their interactions
users_in_both = set(train_data['uid']).intersection(set(test_data['uid']))
print(f"Users in both train and test: {len(users_in_both)}")

Users in both train and test: 19080


In [36]:
train_data.to_csv('train_set.csv', index=False)
test_data.to_csv('test_set.csv', index=False)

In [37]:
print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (42001, 13)
Test shape: (19080, 13)


In [ ]:
# can ignore from here

In [24]:
date_min = pd.to_datetime(merge_df.timestamp, unit='ms').min()
date_max = pd.to_datetime(merge_df.timestamp, unit='ms').max()
print("Date min", date_min)
print("Date max", date_max)
date_gap = (date_max - date_min) // (26 * 2)
merge_df['time'] = ((pd.to_datetime(merge_df['timestamp'], unit='ms') - date_min) // date_gap).astype(int)

sample_df['time'] = ((pd.to_datetime(merge_df['timestamp'], unit='ms') - date_min) // date_gap).astype(int)

Date min 1997-11-12 01:37:58
Date max 2023-09-08 22:06:04.465000


/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_19645/3762797036.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df['time'] = ((pd.to_datetime(merge_df['timestamp'], unit='ms') - date_min) // date_gap).astype(int)


In [25]:
merge_df.shape
merge_df.head(3)

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,rating,iid,parent_iid,uid,timestamp,reviews,main_category,title,average_rating,rating_number,description,categories,label,time
0,1.0,B000002X4C,B000002X4C,AGDQO4VIXXMHZMFRFHAHO2PNCYRQ,1.420266e+12,Definitely the weakest of the three albums SWV...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",0,34
1,1.0,B000002X4C,B000002X4C,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,1.379363e+12,Unlike It's About Time and New Beginning this ...,Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",0,31
2,4.0,B000002X4C,B000002X4C,AHAIZWWINONHYU7A3FVCEH75R4CA,9.534267e+11,"Their third and final album as a group, &quot;...",Digital Music,Release Some Tension,4.601562,112,Swv ~ Release Some Tension,"CDs & Vinyl, Dance & Electronic, House",1,4


In [26]:
sample_df.shape

(100000, 14)

In [27]:
# # train/valid/test split using time for merge_df
# rating_train = merge_df[merge_df['time'] < 50].copy()
# rating_valid_test = merge_df[merge_df['time'] == 50].copy().sort_values('timestamp')
# n = len(rating_valid_test) // 2
# rating_valid = rating_valid_test.iloc[:n].copy()
# rating_test = rating_valid_test.iloc[n:].copy()

# # train/valid/test split using time for merge_df
rating_train = merge_df[merge_df['time'].isin(range(2 * 26 - 2))].copy()
rating_valid_test = merge_df[merge_df['time'] == 2 * 26 - 2].copy()
rating_valid_test.sort_values(by='timestamp', inplace=True)
n = rating_valid_test.shape[0] // 2
rating_valid = rating_valid_test.iloc[:n].copy()
rating_test = rating_valid_test.iloc[n:].copy()

In [28]:
print("Train shape:", rating_train.shape)
print("Valid shape:", rating_valid.shape)
print("Test shape:", rating_test.shape)

Train shape: (2099297, 14)
Valid shape: (11438, 14)
Test shape: (11439, 14)


In [29]:
# train/valid/test split using time for sample_df
rating_train_sample = sample_df[sample_df['time'].isin(range(2 * 26 - 2))].copy()
rating_test_sample = sample_df[sample_df['time'] == 2 * 26 - 2].copy()
rating_test_sample.sort_values(by='timestamp', inplace=True)
# n = rating_valid_test_sample.shape[0] // 2
# rating_valid_sample = rating_valid_test_sample.iloc[:n].copy()
# rating_test_sample = rating_valid_test_sample.iloc[n:].copy()
# print("Valid shape:", rating_valid_sample.shape)
print("Test shape:", rating_test_sample.shape)

Test shape: (1110, 14)


In [30]:
# check cold-start and remove unseen users/items in val/test
train_user_sample = rating_train_sample['uid'].unique()
train_item_sample = rating_train_sample['iid'].unique()
# rating_valid['cold_start'] = (~rating_valid['uid'].isin(train_user)).astype(int)
# rating_test['cold_start'] = (~rating_test['uid'].isin(train_user)).astype(int)

# rating_valid_sample = rating_valid_sample[rating_valid_sample['uid'].isin(train_user_sample) & rating_valid_sample['iid'].isin(train_item_sample)]
rating_test_sample = rating_test_sample[rating_test_sample['uid'].isin(train_user_sample) & rating_test_sample['iid'].isin(train_item_sample)]

In [31]:
def leave_one_out_split(merge_df, min_user_interactions=3):
    """Leave last interaction for test - good balance of realism and evaluation"""
    
    # Sort by timestamp
    merge_df = merge_df.sort_values(['uid', 'timestamp'])
    
    # For each user, take last interaction as test
    test_data = merge_df.groupby('uid').tail(1).copy()
    
    # Training data is all other interactions
    train_data = merge_df[~merge_df.index.isin(test_data.index)].copy()
    
    # Filter users with enough interactions
    user_train_counts = train_data['uid'].value_counts()
    qualified_users = user_train_counts[user_train_counts >= min_user_interactions].index
    
    train_data = train_data[train_data['uid'].isin(qualified_users)]
    test_data = test_data[test_data['uid'].isin(qualified_users)]
    
    print(f"Leave-One-Out Split Results:")
    print(f"Train: {train_data.shape[0]} interactions, {train_data['uid'].nunique()} users")
    print(f"Test: {test_data.shape[0]} interactions, {test_data['uid'].nunique()} users")
    
    # Check item overlap
    train_items = set(train_data['iid'].unique())
    test_items = set(test_data['iid'].unique())
    overlap = train_items.intersection(test_items)
    
    print(f"Item overlap: {len(overlap)}/{len(test_items)} ({len(overlap)/len(test_items):.1%})")
    
    return train_data, test_data

# Usage
train_data, test_data = leave_one_out_split(merge_df)

Leave-One-Out Split Results:
Train: 1984114 interactions, 140856 users
Test: 140856 interactions, 140856 users
Item overlap: 69453/80613 (86.2%)


In [27]:
# identify cold-start users- val, test sets
cold_start_users = set(rating_test_sample['uid'].unique()) - set(train_user_sample)
print("Train shape:", rating_train_sample.shape)
# print("Valid shape:", rating_valid_sample.shape)
print("Test shape:", rating_test_sample.shape)
print("# Cold-start users:", len(cold_start_users))

Train shape: (98597, 14)
Test shape: (172, 14)
# Cold-start users: 0


In [28]:
# Save the datasets to files
rating_train_sample.to_csv('train_set.csv', index=False)
rating_test_sample.to_csv('test_set.csv', index=False)

In [26]:
# import pandas as pd
# load_path = "../../data/amazon/samples"

# sample_10k = pd.read_csv(f"{load_path}/sample_10k.csv")
# sample_100k = pd.read_csv(f"{load_path}/sample_100k.csv")

# print(f"10k samples loaded with {len(sample_10k)} rows")
# print(f"100k samples loaded with {len(sample_100k)} rows")

In [ ]:
# can stop here as having a lightfm module

In [ ]:
print(fitted_data['iid'].nunique())
print(len(dataset.mapping()[1]))

In [37]:
from lightfm import LightFM
from lightfm.data import Dataset
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer
import uuid

def fit_and_train(data, learning_rate, epochs):
    dataset = Dataset()
    # Fit on all possible users and items first
    dataset.fit(users=data['uid'].unique(), 
               items=data['iid'].unique())
    
    # Verify mappings
    user_id_map, _, item_id_map, _ = dataset.mapping()
    print(f"Users in mapping: {len(user_id_map)}, Items in mapping: {len(item_id_map)}")
    
    # build interaction matrices
    interactions, weights = dataset.build_interactions(
        ((row['uid'], row['iid'], row['label']) for _, row in data.iterrows()))
    
    print(f"Interaction matrix shape: {interactions.shape}")
    
    model = LightFM(loss='warp', learning_rate=learning_rate, 
                   item_alpha=1e-6, user_alpha=1e-6)
    model.fit(interactions, epochs=epochs, num_threads=4)
    return model, dataset, interactions, data, weights

In [38]:
def get_top_k(model, dataset, k, data):
    user_id_map, _, item_id_map, _ = dataset.mapping()
    index_to_item_id = {v: k for k, v in item_id_map.items()}
    
    all_user_ids = data['uid'].unique()
    res = {}
    
    for user in tqdm(all_user_ids):
        if user not in user_id_map:
            print(f"User {user} not in mapping")
            continue
            
        user_index = user_id_map[user]
        n_items = len(item_id_map)
        scores = model.predict(user_index, np.arange(n_items))
        
        top_indices = np.argsort(-scores)[:k]
        
        recs = []
        for idx in top_indices:
            raw_item_id = index_to_item_id[idx]
            item_rows = data[data['iid'] == raw_item_id]
            if not item_rows.empty:
                item_row = item_rows.iloc[0]
                recs.append({
                    'iid': raw_item_id,
                    'title': item_row.get('title', ''),
                    'categories': item_row.get('categories', ''),
                    'description': item_row.get('description', ''),
                    'score': float(scores[idx])
                })
        
        res[user] = recs
    
    return res

In [39]:
model, dataset, interactions, fitted_data, weights = fit_and_train(rating_train_sample, learning_rate=0.01, epochs=50)

Users in mapping: 57297, Items in mapping: 63723
Interaction matrix shape: (57297, 63723)


In [40]:
top_k_results = get_top_k(model=model, dataset=dataset, k=50, data=fitted_data)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57297/57297 [12:47:25<00:00,  1.24it/s]


In [41]:
# After training, check loss
train_loss = model.fit(interactions, epochs=50, num_threads=4, 
                      verbose=True, sample_weight=weights)
print("Training loss:", train_loss)

Epoch: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:04<00:00, 10.18it/s]

Training loss: <lightfm.lightfm.LightFM object at 0x53e3a52d0>


In [59]:
# print(f"Sample user recommendations: {list(top_k_results.values())[:1]}")

Sample user recommendations: [[{'iid': 'B000002U82', 'title': 'Dark Side Of The Moon', 'categories': 'CDs & Vinyl, Rock, Progressive, Progressive Rock', 'description': 'Product description, DARK SIDE OF THE MOON was a benchmark record. It turned the musical world on its ear with a hitherto unseen combination of sounds, and changed things considerably for Pink Floyd. For this project, Pink Floyd resurrected older and unfinished numbers, some of which came from the multitude of soundtracks the band members had previously worked on. The film ZABRISKIE POINT, a study of American materialism from a foreigner\'s perspective, provided /Us and Them" (originally titled ""The Violence Sequence""). Waters rewrote ""Breathe"" after its appearance on his and avant-garde composer Ron Geesin\'s score for THE BODY" Amazon.com Dark Side of the Moon, originally released in 1973, is one of those albums that is discovered anew by each generation of rock listeners. This complex, often psychedelic music wor

In [46]:
print("Top 50 recommendations for first 3 users:\n")

for i, (user_id, recs) in enumerate(top_k_results.items()):
    if i >= 3:
        break
    print(f"user_id: {user_id}  top 50 is:")
    for rec in recs:
        print(f"  'iid': {rec['iid']}, 'title': {rec['title']}, 'categories': {rec['categories']}, 'description': {rec['description']}, 'score': {rec['score']}")
    print()

Top 50 recommendations for first 5 users:

user_id: AE4QWKT4VX3DUVNTZM2OHJ2K7YMQ  top 50 is:
  'iid': B000002U82, 'title': Dark Side Of The Moon, 'categories': CDs & Vinyl, Rock, Progressive, Progressive Rock, 'description': Product description, DARK SIDE OF THE MOON was a benchmark record. It turned the musical world on its ear with a hitherto unseen combination of sounds, and changed things considerably for Pink Floyd. For this project, Pink Floyd resurrected older and unfinished numbers, some of which came from the multitude of soundtracks the band members had previously worked on. The film ZABRISKIE POINT, a study of American materialism from a foreigner's perspective, provided /Us and Them" (originally titled ""The Violence Sequence""). Waters rewrote ""Breathe"" after its appearance on his and avant-garde composer Ron Geesin's score for THE BODY" Amazon.com Dark Side of the Moon, originally released in 1973, is one of those albums that is discovered anew by each generation of roc

In [55]:
# import pandas as pd
# import pyarrow as pa
# import pyarrow.parquet as pq
# import os

# def save_initial_recs_parquet(results, file_path):
#     records = []
#     for user_id, recs in results.items():
#         records.append({
#             'user_id': user_id,
#             'recommendations': recs
#         })
    
#     df = pd.DataFrame(records)

#     recommendation_schema = pa.schema([
#         ('iid', pa.string()),
#         ('title', pa.string()),
#         ('categories', pa.string()),
#         ('description', pa.string()),
#         ('score', pa.float32())
#     ])

#     recommendations_array = []
#     for rec_list in df['recommendations']:
#         struct_list = []
#         for rec in rec_list:
#             struct_list.append({
#                 'iid': rec['iid'],
#                 'title': rec.get('title', ''),
#                 'categories': rec.get('categories', ''),
#                 'description': rec.get('description', ''),
#                 'score': float(rec['score'])
#             })
#         recommendations_array.append(struct_list)

#     user_ids = pa.array(df['user_id'])
#     recommendations = pa.array(recommendations_array, 
#                              pa.list_(pa.struct(recommendation_schema)))

#     table = pa.Table.from_arrays(
#         [user_ids, recommendations],
#         schema=pa.schema([
#             ('user_id', pa.string()),
#             ('recommendations', pa.list_(pa.struct(recommendation_schema)))
#         ])
#     )
        
#     pq.write_table(table, file_path, compression = 'snappy')

In [56]:
output_dir = "/Users/vynguyen/Documents/VSCode/Thesis/data/amazon/initial_recs"
os.makedirs(output_dir, exist_ok=True)  # Create directory if doesn't exist

file_path = os.path.join(output_dir, "recommendations.parquet")
save_initial_recs_parquet(top_k_results, file_path=file_path)

In [57]:
import json
import gzip

def save_to_jsonl_gz(results, file_path):
    with gzip.open(file_path, 'wt', encoding='utf-8') as f:
        for user_id, recs in results.items():
            json.dump({
                'user_id': user_id,
                'recommendations': recs
            }, f)
            f.write('\n')  

In [58]:
save_to_jsonl_gz(top_k_results, "recommendations.jsonl.gz")